# Modelo XGBoost de originación — cuaderno unificado

| bloque | venía de | la pregunta que responde |
|---|---|---|
| **A** | V4.2 | ¿qué hiperparámetros encuentra una búsqueda aleatoria normal para cada población, y cuánto aporta cada fuente externa? |
| **B** | V5.1 | si en vez de buscar al azar pruebo a mano cinco ideas concretas (más regularización, más peso a la mora, crecer por hoja, aprender lento…), ¿alguna le gana a lo que encontró A? |
| **C** | V5.2 | si abro el espacio de búsqueda hasta rozar el sobreajuste y uso la validación cruzada como juez, ¿hay una mejora real o A ya estaba en el techo? |

Los tres modelos que se repiten en cada bloque son los mismos de siempre:

- **Interno**: toda la cartera madura, solo variables propias.
- **Datacrédito**: la subpoblación con consulta de buró, con su línea base "solo internas" para medir el aporte del buró.
- **Vinculación digital**: créditos con vinculación digital pero sin Datacrédito, también con su línea base.

**Cómo usarlo.** Las secciones 1 a 3 son obligatorias: configuran, definen funciones y cargan datos. Después cada bloque se puede correr solo. B y C usan los resultados de A como punto de comparación.

## 1. Configuración

Rutas, constantes y el número de combinaciones que prueba cada búsqueda. 

In [ ]:
import pandas as pd, numpy as np, re, glob, time
from pathlib import Path
from sklearn.base import clone
from sklearn.model_selection import (train_test_split, RandomizedSearchCV, StratifiedKFold,
                                     cross_val_score)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from scipy.stats import randint, uniform
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
    brier_score_loss, confusion_matrix, f1_score, precision_score, recall_score,
    balanced_accuracy_score, matthews_corrcoef, precision_recall_curve, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# --- Rutas ---
BASE_DIR     = Path(r"C:\Users\anainformacion1\Desktop\Rohan\Archivos_Modelo")
DATA_INTERNA = BASE_DIR / "interim" / "base_xgboost.csv"
DATA_DC      = BASE_DIR / "interim" / "base_xgboost_final.csv"
VINC_GLOB    = str(BASE_DIR / "Vinculaciondigital*.csv")

OUTPUT_DIR = BASE_DIR / "Unificado"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Default_Altura"; FECHA_T0 = "FechaColocacion"; RANDOM_STATE = 42

# Anonimización de la cédula de vinculación digital, tal como estaba en las tres versiones.
# OJO: solo traduce 0,1,2,3,5,8. Si la base interna se anonimizó con el mapa completo
# (0→D 1→Q 2→S 3→F 4→G 5→H 6→K 7→L 8→J 9→P), las cédulas con 4, 6, 7 o 9 no van a cruzar.
# Vale la pena revisarlo antes de confiar en el tamaño del segmento de vinculación digital.
MAP_ID = str.maketrans({'0':'D','1':'Q','2':'S','3':'F','5':'H','8':'J'})

# --- Validación cruzada (igual en A y C) ---
CV_FOLDS = 5
N_JOBS   = 1      # 1 evita saturar disco/RAM con bases grandes

# --- Tamaño de cada búsqueda ---
N_ITER_A = 40     # bloque A: búsqueda "normal"
N_ITER_C = 120    # bloque C: búsqueda amplia

# Nombres de los tres modelos, usados como llave en todo el cuaderno
MODELOS = ["Interno", "Datacrédito", "Vinculación Digital"]

print(f"Salidas en: {OUTPUT_DIR}")
print(f"Bloque A: {N_ITER_A} x {CV_FOLDS} = {N_ITER_A*CV_FOLDS} entrenamientos por búsqueda")
print(f"Bloque C: {N_ITER_C} x {CV_FOLDS} = {N_ITER_C*CV_FOLDS} entrenamientos por búsqueda")

## 2. Funciones compartidas

Todo lo que los tres bloques usan igual: métricas, filtro de madurez, control de fuga, preprocesamiento, las mejoras de datos y las herramientas de evaluación.

La función de búsqueda es una sola para A y C. Lo único que cambia entre ellos es el espacio de hiperparámetros y el número de combinaciones, que se le pasan como argumento. Si recibe una referencia (los resultados de A), además dice si la mejora supera el ruido de la validación cruzada.

In [ ]:
# ---------- Métricas ----------
def ks_stat(y, p):
    fpr, tpr, _ = roc_curve(y, p); return float(np.max(tpr - fpr))

def metricas(y, p, nombre):
    auc = roc_auc_score(y, p)
    return {"modelo": nombre, "AUC": round(auc,4), "Gini": round(2*auc-1,4),
            "KS": round(ks_stat(y,p),4), "AUC_PR": round(average_precision_score(y,p),4),
            "Brier": round(brier_score_loss(y,p),4), "n_test": len(y),
            "default_%": round(float(np.mean(y))*100,1)}

# ---------- Filtro de madurez ----------
def aplicar_madurez(df):
    """Conserva solo créditos cuya ventana de observación ya se completó."""
    df = df.copy()
    for c in ["FechaColocacion", "FechaCorte", "RequestDate"]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    corte = df["FechaCorte"].max()
    fin = df[FECHA_T0] + df["Altura_observacion"].astype(int).apply(lambda m: pd.DateOffset(months=m))
    return df[fin <= corte].copy()

# ---------- Control de fuga ----------
def detectar_leakage(df, extra_drop=None):
    """Excluye identificadores, fechas, objetivo, variables posteriores a la originación y constantes."""
    IDS = ["Factura", "Cedula"]; FECHAS = ["FechaColocacion", "FechaCorte", "RequestDate"]
    LEAK = ["DiasMoraEfectivo","DiasAtraso","DiasAtrasoReporte","DiasAtrasoReporteAcum",
            "CuotasCanceladas","AlturaColocacion_meses","estado_credito", TARGET, "FechaCorte"]
    PAT = re.compile(r"(mora|atraso|default|vencid|castig|recuper|posterior|futur|"
                     r"saldo.?venc|dias.?mora)", re.IGNORECASE)
    WHITELIST_T0 = {"SaldoTotalEnMora"}
    fuga = set()
    for c in df.columns:
        if c in IDS or c in FECHAS or c == TARGET: continue
        if c in LEAK: fuga.add(c); continue
        if PAT.search(c) and c not in WHITELIST_T0: fuga.add(c)
    EXC = set(IDS) | set(FECHAS) | {TARGET} | fuga | {"AnoColoc","Altura_observacion","maduro"}
    if extra_drop: EXC |= set(extra_drop)
    const = [c for c in df.columns if c not in (set(IDS)|set(FECHAS)|{TARGET})
             and df[c].nunique(dropna=True) <= 1]
    EXC |= set(const)
    pred = [c for c in df.columns if c not in EXC]
    num = [c for c in pred if pd.api.types.is_numeric_dtype(df[c])]
    cat = [c for c in pred if not pd.api.types.is_numeric_dtype(df[c])]
    return num, cat

def separar_tipos(df, cols):
    num = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    cat = [c for c in cols if not pd.api.types.is_numeric_dtype(df[c])]
    return num, cat

# ---------- Preprocesamiento ----------
def construir_preprocesador(num_cols, cat_cols):
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=50))]), cat_cols)])

# ---------- Mejoras de datos ----------
def aplicar_mejoras(df):
    """Flags de nulos/atípicos, corrección de errores reales, agrupación de categorías raras y estacionalidad."""
    df = df.copy()
    for c in ["ValorColocacion","Cuotas","ScoreRiesgo","IngresoDisponibleCliente",
              "EdadCliente","SaldoContableNiif"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # 1. Flags: los nulos y atípicos marcan segmentos con default distinto
    if "CALIFICACION" in df.columns:
        df["flag_sin_calificacion"] = df["CALIFICACION"].isna().astype(int)
    if "ScoreRiesgo" in df.columns:
        df["flag_sin_score"] = df["ScoreRiesgo"].isna().astype(int)
        df["flag_score_negativo"] = ((df["ScoreRiesgo"] < 0) & df["ScoreRiesgo"].notna()).astype(int)
    if "IngresoDisponibleCliente" in df.columns:
        df["flag_ingreso_cero"] = (df["IngresoDisponibleCliente"] == 0).astype(int)
    if "ValorColocacion" in df.columns:
        df["flag_colocacion_sobre_max"] = (df["ValorColocacion"] > 6_000_000).astype(int)
    if "Cuotas" in df.columns:
        df["flag_plazo_extendido"] = (df["Cuotas"] > 36).astype(int)
    if {"SaldoContableNiif","ValorColocacion"}.issubset(df.columns):
        df["flag_saldo_mayor_colocacion"] = (
            (df["SaldoContableNiif"] > df["ValorColocacion"]) &
            df["SaldoContableNiif"].notna() & df["ValorColocacion"].notna()).astype(int)

    # 2. Corrección solo de errores reales
    if "EdadCliente" in df.columns:
        df.loc[(df["EdadCliente"] < 18) | (df["EdadCliente"] > 100), "EdadCliente"] = np.nan
    if "IngresoDisponibleCliente" in df.columns:   # unidad: SMMLV
        df.loc[df["IngresoDisponibleCliente"] < 0, "IngresoDisponibleCliente"] = np.nan

    # 3. Categorías con menos de 100 registros se agrupan en "OTRAS"
    for col in ["Ciudad", "Centrocosto"]:
        if col in df.columns:
            vc = df[col].value_counts(dropna=True)
            frec = vc[vc >= 100].index
            df[col] = df[col].where(df[col].isin(frec) | df[col].isna(), "OTRAS")

    # 4. Estacionalidad
    if FECHA_T0 in df.columns:
        f = pd.to_datetime(df[FECHA_T0], errors="coerce")
        df["mes_colocacion"] = f.dt.month
        df["trim_colocacion"] = f.dt.quarter
    return df

# ---------- Búsqueda de hiperparámetros (A y C) ----------
def buscar_hiperparametros(X_train, y_train, num_cols, cat_cols, nombre,
                           param_dist, n_iter, ref=None):
    """Busca por validación cruzada sobre train. El test nunca participa.
    ref: dict con cv_auc y cv_std de otra búsqueda, para decir si la diferencia supera el ruido."""
    t0 = time.time()
    prep = construir_preprocesador(num_cols, cat_cols)
    spw = (y_train == 0).sum() / (y_train == 1).sum()
    base_clf = XGBClassifier(scale_pos_weight=spw, eval_metric="auc", tree_method="hist",
                             random_state=RANDOM_STATE, n_jobs=N_JOBS)
    pipe = Pipeline([("prep", prep), ("clf", base_clf)])
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(pipe, param_distributions=param_dist, n_iter=n_iter,
                                scoring="roc_auc", cv=cv, n_jobs=N_JOBS, refit=True,
                                random_state=RANDOM_STATE, verbose=1, return_train_score=True)
    search.fit(X_train, y_train)

    i = search.best_index_
    cv_mean = search.cv_results_["mean_test_score"][i]
    cv_std  = search.cv_results_["std_test_score"][i]
    tr_mean = search.cv_results_["mean_train_score"][i]
    gap = tr_mean - cv_mean
    mins = (time.time()-t0)/60

    print(f"\n── Búsqueda: {nombre} ({mins:.1f} min) ──")
    print(f"  AUC CV      : {cv_mean:.4f} ± {cv_std:.4f}")
    print(f"  AUC train   : {tr_mean:.4f}")
    print(f"  Gap train-CV: {gap:.4f}  ({'OK' if gap < 0.03 else 'CUIDADO: posible sobreajuste'})")
    best = {k.replace("clf__",""): v for k,v in search.best_params_.items()}
    print("  Mejores hiperparámetros:")
    for k,v in best.items():
        print(f"    {k}: {round(v,5) if isinstance(v,float) else v}")
    diag = {"modelo": nombre, "cv_auc": round(float(cv_mean),4), "cv_std": round(float(cv_std),4),
            "train_auc": round(float(tr_mean),4), "gap_train_cv": round(float(gap),4),
            "minutos": round(mins,1),
            **{k: (round(v,5) if isinstance(v,float) else v) for k,v in best.items()}}

    if ref is not None:
        delta_cv = cv_mean - ref["cv_auc"]
        umbral = ref["cv_std"]
        supera = delta_cv > umbral
        print(f"\n  >>> Contra la referencia: CV {cv_mean:.4f} vs {ref['cv_auc']:.4f} "
              f"(delta {delta_cv:+.4f}, ruido ±{umbral:.4f})")
        print(f"      {'MEJORA REAL: supera el ruido' if supera else 'Dentro del ruido: no es mejora real'}")
        diag["delta_cv_vs_ref"] = round(float(delta_cv),4)
        diag["supera_ruido"] = "SI" if supera else "no"
    return search.best_estimator_, best, diag

# ---------- Entrenar una configuración fija (B) ----------
def entrenar_config(X_train, y_train, X_test, y_test, num_cols, cat_cols, params,
                    nombre_config, nombre_modelo):
    """Entrena una configuración fija y la evalúa en test."""
    t0 = time.time()
    prep = construir_preprocesador(num_cols, cat_cols)
    params = dict(params)
    spw = params.pop("scale_pos_weight", "auto")      # "auto" = n_neg / n_pos del train
    if spw == "auto":
        spw = (y_train == 0).sum() / (y_train == 1).sum()
    clf = XGBClassifier(scale_pos_weight=spw, eval_metric="auc", tree_method="hist",
                        random_state=RANDOM_STATE, n_jobs=-1, **params)
    pipe = Pipeline([("prep", prep), ("clf", clf)])
    pipe.fit(X_train, y_train)
    p = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, p)
    yh = (p >= umbral_f1(y_test, p)).astype(int)
    mins = (time.time()-t0)/60
    fila = {"modelo": nombre_modelo, "config": nombre_config,
            "AUC": round(auc,4), "Gini": round(2*auc-1,4), "KS": round(ks_stat(y_test,p),4),
            "AUC_PR": round(average_precision_score(y_test,p),4),
            "Brier": round(brier_score_loss(y_test,p),4),
            "F1": round(f1_score(y_test,yh),4), "recall": round(recall_score(y_test,yh),4),
            "precision": round(precision_score(y_test,yh),4),
            "MCC": round(matthews_corrcoef(y_test,yh),4), "minutos": round(mins,1)}
    print(f"  [{nombre_config:<24}] AUC={auc:.4f} F1={fila['F1']:.4f} "
          f"recall={fila['recall']:.4f} MCC={fila['MCC']:.4f} ({mins:.1f} min)")
    return fila, p

# ---------- Umbral y matrices de confusión ----------
def umbral_f1(y_true, p):
    """Umbral que maximiza F1. Ojo: aquí se calcula sobre el mismo conjunto que se evalúa."""
    prec, rec, ths = precision_recall_curve(y_true, p)
    f1g = np.divide(2*prec*rec, prec+rec, out=np.zeros_like(prec), where=(prec+rec)>0)
    return float(ths[int(np.argmax(f1g[:-1]))])

def matriz_f1(y_true, p, nombre):
    th = umbral_f1(y_true, p)
    yh = (p >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yh).ravel()
    print(f"\n=== {nombre} | umbral óptimo F1 = {th:.3f} ===")
    print(f"              Pred 0     Pred 1")
    print(f"  Real 0   {tn:>9,} {fp:>10,}")
    print(f"  Real 1   {fn:>9,} {tp:>10,}")
    print(f"  F1={f1_score(y_true,yh):.4f}  Precision={precision_score(y_true,yh):.4f}  "
          f"Recall={recall_score(y_true,yh):.4f}  Bal.Acc={balanced_accuracy_score(y_true,yh):.4f}  "
          f"MCC={matthews_corrcoef(y_true,yh):.4f}")
    return {"modelo": nombre, "umbral_opt": round(th,3), "TN": int(tn), "FP": int(fp),
            "FN": int(fn), "TP": int(tp), "F1": round(f1_score(y_true,yh),4),
            "precision": round(precision_score(y_true,yh),4), "recall": round(recall_score(y_true,yh),4),
            "balanced_acc": round(balanced_accuracy_score(y_true,yh),4),
            "MCC": round(matthews_corrcoef(y_true,yh),4)}

def graficar_matrices(casos, titulo_extra=""):
    """casos: lista de (y_true, p, nombre)."""
    cmaps = ["Blues", "Greens", "Oranges"]
    fig, ax = plt.subplots(1, len(casos), figsize=(5*len(casos), 4.2))
    for a, (yt, pr, nom), cmap in zip(np.atleast_1d(ax), casos, cmaps):
        th = umbral_f1(yt, pr)
        cm = confusion_matrix(yt, (pr >= th).astype(int))
        ConfusionMatrixDisplay(cm, display_labels=["No def.","Default"]).plot(
            ax=a, cmap=cmap, colorbar=False, values_format=",d")
        a.set_title(f"{nom} (umbral {th:.2f}){titulo_extra}")
    plt.tight_layout(); plt.show()

# ---------- Validación con varias semillas (C) ----------
def validar_multiseed(estimator, X, y, semillas=(7, 123, 2024)):
    """Reentrena la configuración en varios splits de train y reporta el AUC de cada uno."""
    aucs = []
    for sd in semillas:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, random_state=sd, stratify=y)
        m = clone(estimator); m.fit(Xtr, ytr)
        aucs.append(roc_auc_score(yte, m.predict_proba(Xte)[:,1]))
    return float(np.mean(aucs)), float(np.std(aucs)), aucs

print("Funciones listas.")

## 3. Datos y particiones (comunes a los tres bloques)

Esto se carga una sola vez. Las tres versiones originales hacían exactamente lo mismo aquí, con la misma semilla, así que los tres bloques trabajan sobre los mismos train y test. Eso es lo que permite compararlos de forma limpia.

El split es aleatorio estratificado 60/20/20 con semilla 42. El 20% de validación (`valid_*`) se separa pero ningún bloque lo usa: la búsqueda se valida con validación cruzada sobre train y el test se toca solo al final.

### 3.1 Cartera interna

In [ ]:
df_int = pd.read_csv(DATA_INTERNA, sep=";", encoding="utf-8-sig", low_memory=False)
df_int = aplicar_mejoras(df_int)
df_int = aplicar_madurez(df_int)
print(f"Interno maduro: {len(df_int):,} créditos | default {df_int[TARGET].mean()*100:.1f}%")

num_i, cat_i = detectar_leakage(df_int)
FEAT_I = num_i + cat_i
print(f"Variables: {len(num_i)} numéricas + {len(cat_i)} categóricas = {len(FEAT_I)}")

df_int = df_int.sort_values(FECHA_T0).reset_index(drop=True)
tr_df, test_i = train_test_split(df_int, test_size=0.20, random_state=RANDOM_STATE, stratify=df_int[TARGET])
train_i, valid_i = train_test_split(tr_df, test_size=0.25, random_state=RANDOM_STATE, stratify=tr_df[TARGET])
print(f"Train {len(train_i):,} | Valid {len(valid_i):,} | Test {len(test_i):,}")

Xtr_i, ytr_i = train_i[FEAT_I], train_i[TARGET].astype(int)
Xte_i, yte_i = test_i[FEAT_I],  test_i[TARGET].astype(int)

### 3.2 Vinculación digital

Une los archivos 2024 a 2026, resuelve que la columna del documento a veces viene como `NumeroDocumento_num` y a veces como `NumeroDocumento_Num`, anonimiza y se queda con la solicitud más reciente de cada cliente.

In [ ]:
def money(s):
    return pd.to_numeric(s.astype(str).str.replace(r"[$\s.]","",regex=True)
                          .str.replace(",",".",regex=False), errors="coerce")
def num_coma(s):
    return pd.to_numeric(s.astype(str).str.replace(",",".",regex=False), errors="coerce")

COLS_VD = {"numerodocumento_num":"doc","createdon":"vd_fecha","varacierta":"vd_varacierta",
    "decisiontrust":"vd_decisiontrust","var_grupo_riesgo":"vd_grupo_riesgo","var_quanto":"vd_quanto",
    "var_gasto_familiar":"vd_gasto_familiar","var_disponible":"vd_disponible",
    "cuotasfinancieras":"vd_cuotas_fin","tendenciaingresos":"vd_tendencia_ing",
    "ingreso_total_ult_mes":"vd_ing_mes","promedio_ult_tres_meses":"vd_ing_prom3m",
    "tiempo_total_laborado":"vd_tiempo_lab","calificacionclienteagaval":"vd_calif_agaval",
    "moradata":"vd_mora_data"}
VD_NUM = ["vd_varacierta","vd_decisiontrust","vd_quanto","vd_gasto_familiar","vd_disponible",
          "vd_cuotas_fin","vd_ing_mes","vd_ing_prom3m","vd_tiempo_lab"]
VD_CAT = ["vd_grupo_riesgo","vd_tendencia_ing","vd_calif_agaval","vd_mora_data"]

archivos_vd = sorted(glob.glob(VINC_GLOB))
if not archivos_vd:
    archivos_vd = sorted(str(p) for p in BASE_DIR.rglob("Vinculaciondigital*.csv"))
assert archivos_vd, f"No se encontraron archivos Vinculaciondigital*.csv en {VINC_GLOB}"

frames = []
for arch in archivos_vd:
    d = pd.read_csv(arch, sep=";", encoding="latin-1", dtype=str, low_memory=False)
    d.columns = [c.strip().lower() for c in d.columns]      # resuelve _num vs _Num
    d = d.loc[:, ~d.columns.duplicated()]
    ren = {c: COLS_VD[c] for c in d.columns if c in COLS_VD}
    frames.append(d[list(ren)].rename(columns=ren))
    print(f"  {Path(arch).name}: {len(d):,} filas")

vinc = pd.concat(frames, ignore_index=True)
vinc["doc"] = vinc["doc"].astype(str).str.strip()
vinc = vinc[vinc["doc"].str.match(r"^\d+$", na=False)]
vinc["Cedula"] = vinc["doc"].apply(lambda s: s.translate(MAP_ID))
for c in ["vd_ing_mes","vd_ing_prom3m"]:
    if c in vinc.columns: vinc[c] = money(vinc[c])
for c in ["vd_quanto","vd_gasto_familiar","vd_disponible","vd_cuotas_fin",
          "vd_varacierta","vd_decisiontrust","vd_tiempo_lab"]:
    if c in vinc.columns: vinc[c] = num_coma(vinc[c])
vinc["vd_fecha"] = pd.to_datetime(vinc["vd_fecha"], format="%d/%m/%Y", errors="coerce")
vinc = vinc.sort_values("vd_fecha").drop_duplicates("Cedula", keep="last")
print(f"\nVinculación digital unificada: {len(vinc):,} clientes únicos")

### 3.3 Subpoblación con Datacrédito

Se arman dos juegos de variables sobre la misma población: las del modelo con buró (`FEAT_DC`) y solo las internas (`internas_dc`), que sirven de línea base para medir cuánto aporta el buró.

In [ ]:
df_dc = pd.read_csv(DATA_DC, sep=";", encoding="utf-8-sig", low_memory=False)
df_dc = aplicar_mejoras(df_dc)
df_dc = aplicar_madurez(df_dc)
print(f"Datacrédito maduro: {len(df_dc):,} | default {df_dc[TARGET].mean()*100:.1f}%")

DROP_DC = {"RequestDate","Nodo","TreeNode","Respuesta","Age_bucket","AgeBasedRegistry_bucket"}
num_dc, cat_dc = detectar_leakage(df_dc, extra_drop=DROP_DC)
FEAT_DC = list(dict.fromkeys(num_dc + cat_dc))
n_dc, c_dc = separar_tipos(df_dc, FEAT_DC)

internas_dc = [c for c in FEAT_I if c in df_dc.columns]
n_int_dc, c_int_dc = separar_tipos(df_dc, internas_dc)

df_dc = df_dc.sort_values(FECHA_T0).reset_index(drop=True)
tr_dc_df, test_dc = train_test_split(df_dc, test_size=0.20, random_state=RANDOM_STATE, stratify=df_dc[TARGET])
train_dc, valid_dc = train_test_split(tr_dc_df, test_size=0.25, random_state=RANDOM_STATE, stratify=tr_dc_df[TARGET])
print(f"Train {len(train_dc):,} | Valid {len(valid_dc):,} | Test {len(test_dc):,}")

ytr_dc, yte_dc = train_dc[TARGET].astype(int), test_dc[TARGET].astype(int)

### 3.4 Segmento de vinculación digital sin Datacrédito

Es el segmento donde la vinculación digital aporta algo propio: créditos que tienen vinculación digital pero no consulta de buró. Igual que con Datacrédito, hay línea base solo con internas.

In [ ]:
ced_dc = set(pd.read_csv(DATA_DC, sep=";", encoding="utf-8-sig", dtype=str,
                         usecols=["Cedula"])["Cedula"].dropna())

df_vd = df_int.merge(vinc[["Cedula"] + VD_NUM + VD_CAT], on="Cedula", how="left")
tiene_vd = df_vd[VD_NUM + VD_CAT].notna().any(axis=1)
tiene_dc = df_vd["Cedula"].isin(ced_dc)
seg = df_vd[tiene_vd & ~tiene_dc].copy()
print(f"Segmento VD sin DC: {len(seg):,} créditos | default {seg[TARGET].mean()*100:.1f}%")

vd_num_ok = [c for c in VD_NUM if c in seg.columns]
vd_cat_ok = [c for c in VD_CAT if c in seg.columns]
FEAT_VD = FEAT_I + vd_num_ok + vd_cat_ok
n_vd, c_vd = num_i + vd_num_ok, cat_i + vd_cat_ok

tr_vd_df, test_vd = train_test_split(seg, test_size=0.20, random_state=RANDOM_STATE, stratify=seg[TARGET])
train_vd, valid_vd = train_test_split(tr_vd_df, test_size=0.25, random_state=RANDOM_STATE, stratify=tr_vd_df[TARGET])
print(f"Train {len(train_vd):,} | Valid {len(valid_vd):,} | Test {len(test_vd):,}")

ytr_vd, yte_vd = train_vd[TARGET].astype(int), test_vd[TARGET].astype(int)

Un atajo para no repetir lo mismo en cada bloque: para cada modelo, qué datos y qué variables usa.

In [ ]:
# modelo -> (X_train, y_train, X_test, y_test, numéricas, categóricas)
POBLACIONES = {
    "Interno":             (Xtr_i, ytr_i, Xte_i, yte_i, num_i, cat_i),
    "Datacrédito":         (train_dc[FEAT_DC], ytr_dc, test_dc[FEAT_DC], yte_dc, n_dc, c_dc),
    "Vinculación Digital": (train_vd[FEAT_VD], ytr_vd, test_vd[FEAT_VD], yte_vd, n_vd, c_vd),
}
# Líneas base "solo internas" para medir el aporte de cada fuente externa
BASES = {
    "Datacrédito":         (train_dc[internas_dc], ytr_dc, test_dc[internas_dc], yte_dc, n_int_dc, c_int_dc),
    "Vinculación Digital": (train_vd[FEAT_I], ytr_vd, test_vd[FEAT_I], yte_vd, num_i, cat_i),
}
print("Poblaciones listas:", list(POBLACIONES))

---

# Bloque A — Búsqueda aleatoria por población 

**Objetivo.** Encontrar, para cada uno de los tres modelos, los hiperparámetros que mejor funcionan en su propia población, y medir de paso cuánto aporta cada fuente externa.

Lo que hace, paso a paso:

1. Corre una búsqueda aleatoria de `N_ITER_A` combinaciones con validación cruzada de 5 folds sobre train, por separado para cada población. Así se ve si los óptimos cambian según la población.
2. Para Datacrédito y vinculación digital corre además una búsqueda para la línea base "solo internas" en esa misma población. La diferencia de AUC entre las dos es el aporte (lift) de la fuente externa. Se mide siempre dentro de la misma población, nunca contra la cartera completa, que tiene otra tasa de default.
3. Reporta el diagnóstico de sobreajuste (gap entre train y validación cruzada) y la desviación entre folds, que sirve como umbral de ruido.
4. Toca el test una sola vez, al final, para reportar desempeño.

In [ ]:
# Espacio de búsqueda del bloque A (igual para las tres poblaciones)
PARAM_DIST_A = {
    "clf__n_estimators":     randint(300, 900),
    "clf__max_depth":        randint(3, 8),
    "clf__learning_rate":    uniform(0.01, 0.19),
    "clf__subsample":        uniform(0.6, 0.4),
    "clf__colsample_bytree": uniform(0.6, 0.4),
    "clf__min_child_weight": randint(1, 15),
    "clf__gamma":            uniform(0.0, 0.6),
    "clf__reg_lambda":       uniform(0.5, 6.0),
    "clf__reg_alpha":        uniform(0.0, 2.0),
}

def correr_busqueda(modelo, datos, param_dist, n_iter, etiqueta, ref=None):
    Xtr, ytr, Xte, yte, num, cat = datos
    est, best, diag = buscar_hiperparametros(Xtr, ytr, num, cat, etiqueta, param_dist, n_iter, ref=ref)
    p = est.predict_proba(Xte)[:, 1]
    return {"estimador": est, "best": best, "diag": diag, "p": p, "m": metricas(yte, p, etiqueta)}

resA, baseA = {}, {}

### A.1 Modelo interno

La búsqueda más pesada del bloque: train de varios cientos de miles de filas por 5 folds por `N_ITER_A` combinaciones. 

In [ ]:
resA["Interno"] = correr_busqueda("Interno", POBLACIONES["Interno"], PARAM_DIST_A, N_ITER_A, "A · Interno")
print("\nMétricas test:", resA["Interno"]["m"])

### A.2 Datacrédito y su línea base

In [ ]:
baseA["Datacrédito"] = correr_busqueda("Datacrédito", BASES["Datacrédito"], PARAM_DIST_A, N_ITER_A,
                                       "A · Solo internas (población DC)")
resA["Datacrédito"]  = correr_busqueda("Datacrédito", POBLACIONES["Datacrédito"], PARAM_DIST_A, N_ITER_A,
                                       "A · Datacrédito")
print("\nMétricas test:", resA["Datacrédito"]["m"])
print(f">>> Aporte de Datacrédito: {resA['Datacrédito']['m']['AUC'] - baseA['Datacrédito']['m']['AUC']:+.4f}")

### A.3 Vinculación digital y su línea base

In [ ]:
baseA["Vinculación Digital"] = correr_busqueda("Vinculación Digital", BASES["Vinculación Digital"],
                                               PARAM_DIST_A, N_ITER_A, "A · Solo internas (segmento VD)")
resA["Vinculación Digital"]  = correr_busqueda("Vinculación Digital", POBLACIONES["Vinculación Digital"],
                                               PARAM_DIST_A, N_ITER_A, "A · Vinculación Digital")
print("\nMétricas test:", resA["Vinculación Digital"]["m"])
print(f">>> Aporte de vinculación digital: "
      f"{resA['Vinculación Digital']['m']['AUC'] - baseA['Vinculación Digital']['m']['AUC']:+.4f}")

### A.4 ¿Los óptimos cambian según la población?

Si los hiperparámetros salen muy distintos entre poblaciones, vale la pena buscarlos por separado. Si salen parecidos, el modelo es robusto a la configuración y bastaría con unos compartidos. Una diferencia de AUC solo cuenta si supera la desviación de la validación cruzada.

In [ ]:
diagA = pd.DataFrame([resA["Interno"]["diag"],
                      baseA["Datacrédito"]["diag"], resA["Datacrédito"]["diag"],
                      baseA["Vinculación Digital"]["diag"], resA["Vinculación Digital"]["diag"]])
cols_orden = ["modelo","cv_auc","cv_std","train_auc","gap_train_cv","minutos",
              "n_estimators","max_depth","learning_rate","subsample","colsample_bytree",
              "min_child_weight","gamma","reg_lambda","reg_alpha"]
diagA = diagA[[c for c in cols_orden if c in diagA.columns]]
display(diagA)
diagA.to_csv(OUTPUT_DIR / "A_hiperparametros.csv", sep=";", encoding="utf-8-sig", index=False)

# La vara para los bloques B y C
REF_A = {m: {"cv_auc":   resA[m]["diag"]["cv_auc"],
             "cv_std":   resA[m]["diag"]["cv_std"],
             "test_auc": resA[m]["m"]["AUC"],
             "params":   resA[m]["best"]} for m in MODELOS}
print("Referencia del bloque A guardada en REF_A.")

### A.5 Resumen, aportes y cascada

Son tres modelos sobre tres poblaciones distintas, así que su nivel absoluto de AUC no se compara entre sí. Lo que sí se compara es el aporte de cada fuente dentro de su población.


In [ ]:
resumenA = pd.DataFrame([resA[m]["m"] for m in MODELOS])
resumenA = resumenA[["modelo","n_test","default_%","AUC","Gini","KS","AUC_PR","Brier"]]
display(resumenA)
resumenA.to_csv(OUTPUT_DIR / "A_resumen_modelos.csv", sep=";", encoding="utf-8-sig", index=False)

liftsA = pd.DataFrame([
    {"capa": m, "poblacion": len(POBLACIONES[m][3]),
     "auc_sin": baseA[m]["m"]["AUC"], "auc_con": resA[m]["m"]["AUC"],
     "lift": round(resA[m]["m"]["AUC"] - baseA[m]["m"]["AUC"], 4)}
    for m in ["Datacrédito", "Vinculación Digital"]])
display(liftsA)
liftsA.to_csv(OUTPUT_DIR / "A_lifts.csv", sep=";", encoding="utf-8-sig", index=False)

print("\nCascada:")
print(f"  1. ¿Tiene Datacrédito?        -> Modelo Datacrédito  (AUC {resA['Datacrédito']['m']['AUC']:.3f})")
print(f"  2. ¿VD pero sin Datacrédito?  -> Modelo Vinc.Digital (AUC {resA['Vinculación Digital']['m']['AUC']:.3f})")
print(f"  3. Sin fuente externa         -> Modelo Interno      (AUC {resA['Interno']['m']['AUC']:.3f})")

fig, ax = plt.subplots(figsize=(9,4.5))
b = ax.bar(resumenA["modelo"], resumenA["AUC"], color=["#4472C4","#70AD47","#ED7D31"])
ax.set_ylim(0.70, 0.90); ax.set_ylabel("AUC"); ax.set_title("Bloque A — AUC por modelo (cada uno sobre su población)")
for r,v in zip(b, resumenA["AUC"]): ax.text(r.get_x()+r.get_width()/2, v+0.003, f"{v:.3f}", ha="center", fontsize=11)
plt.xticks(rotation=12, ha="right"); plt.tight_layout(); plt.show()

### A.6 Matrices de confusión

In [ ]:
clasifA = pd.DataFrame([matriz_f1(POBLACIONES[m][3], resA[m]["p"], f"A · {m}") for m in MODELOS])
display(clasifA)
clasifA.to_csv(OUTPUT_DIR / "A_clasificacion.csv", sep=";", encoding="utf-8-sig", index=False)
graficar_matrices([(POBLACIONES[m][3], resA[m]["p"], m) for m in MODELOS])

---

# Bloque B — Cinco configuraciones a mano

**Objetivo.** En lugar de dejar que el azar elija, probar cinco ideas concretas, cada una con una hipótesis detrás, y ver si alguna le gana a lo que encontró A. Aquí entran tres hiperparámetros que A no tocaba: `max_delta_step`, `scale_pos_weight` fijo y `grow_policy`.

Las cinco ideas:

1. **Referencia del bloque A.** Los óptimos que encontró A para ese mismo modelo. Es el grupo de control.
2. **Más regularizada.** Árboles menos profundos y más frenos, para ver si cerrar el gap entre train y validación ayuda a generalizar.
3. **Agresiva en detección.** Más peso a la clase morosa (`scale_pos_weight=4`), para priorizar recall.
4. **Crecimiento por hoja.** `lossguide` con 64 hojas, para capturar patrones más complejos.
5. **Lenta y grande.** Tasa de aprendizaje baja con muchos árboles.

Dos diferencias:_

- La configuración de referencia ya no son números copiados a mano. Se arma con los óptimos que A encontró para cada modelo, así que cada población se compara contra su propio óptimo. Si no corriste A, esa configuración simplemente no aparece.
- El bloque B mencionaba una búsqueda aleatoria rápida como primera fase.

Esto no usa validación cruzada: cada configuración se entrena una vez en train y se evalúa en test. Por eso es rápido, y por eso mismo una diferencia pequeña contra A puede ser ruido. El bloque C es el que se encarga de eso.

In [ ]:
# Las cuatro configuraciones fijas (la referencia se agrega por modelo, desde REF_A)
CONFIGS_FIJAS = {
    "2_mas_regularizada": dict(
        n_estimators=700, max_depth=5, learning_rate=0.05, subsample=0.7,
        colsample_bytree=0.6, min_child_weight=10, gamma=0.5,
        reg_lambda=8.0, reg_alpha=2.0,
        max_delta_step=2, scale_pos_weight="auto", grow_policy="depthwise"),

    "3_agresiva_recall": dict(
        n_estimators=855, max_depth=7, learning_rate=0.10, subsample=0.85,
        colsample_bytree=0.75, min_child_weight=2, gamma=0.2,
        reg_lambda=3.0, reg_alpha=1.0,
        max_delta_step=3, scale_pos_weight=4.0, grow_policy="depthwise"),

    "4_lossguide": dict(
        n_estimators=800, max_depth=0, learning_rate=0.08, subsample=0.8,
        colsample_bytree=0.7, min_child_weight=3, gamma=0.2,
        reg_lambda=4.0, reg_alpha=1.0, max_leaves=64,
        max_delta_step=1, scale_pos_weight="auto", grow_policy="lossguide"),

    "5_lento_profundo": dict(
        n_estimators=1200, max_depth=6, learning_rate=0.02, subsample=0.8,
        colsample_bytree=0.7, min_child_weight=4, gamma=0.3,
        reg_lambda=5.0, reg_alpha=1.5,
        max_delta_step=1, scale_pos_weight="auto", grow_policy="depthwise"),
}

REF_A = globals().get("REF_A", {})

def configs_para(modelo):
    cfgs = {}
    if modelo in REF_A:
        cfgs["1_referencia_bloque_A"] = {**REF_A[modelo]["params"], "max_delta_step": 0,
                                        "scale_pos_weight": "auto", "grow_policy": "depthwise"}
    else:
        print(f"  (sin bloque A: {modelo} corre sin configuración de referencia)")
    cfgs.update(CONFIGS_FIJAS)
    return cfgs

resultadosB, predB = {}, {}
for modelo in MODELOS:
    Xtr, ytr, Xte, yte, num, cat = POBLACIONES[modelo]
    print(f"\n── {modelo}: configuraciones ──")
    resultadosB[modelo], predB[modelo] = [], {}
    for nombre_cfg, params in configs_para(modelo).items():
        fila, p = entrenar_config(Xtr, ytr, Xte, yte, num, cat, params, nombre_cfg, modelo)
        resultadosB[modelo].append(fila); predB[modelo][nombre_cfg] = p

### B.1 Tabla resumen

Todas las configuraciones en los tres modelos, con su diferencia contra el AUC de test del bloque A cuando existe.

In [ ]:
tablaB = pd.DataFrame([f for m in MODELOS for f in resultadosB[m]])
tablaB["AUC_ref_A"] = tablaB["modelo"].map({m: REF_A[m]["test_auc"] for m in REF_A})
tablaB["delta_vs_A"] = (tablaB["AUC"] - tablaB["AUC_ref_A"]).round(4)
tablaB["supera_A"] = np.where(tablaB["AUC_ref_A"].isna(), "sin ref.",
                              np.where(tablaB["delta_vs_A"] > 0, "SI", "no"))
cols = ["modelo","config","AUC","AUC_ref_A","delta_vs_A","supera_A",
        "Gini","KS","F1","recall","precision","MCC","Brier","minutos"]
tablaB = tablaB[cols]
display(tablaB)
tablaB.to_csv(OUTPUT_DIR / "B_experimento_configs.csv", sep=";", encoding="utf-8-sig", index=False)

mejorB = {m: max(resultadosB[m], key=lambda r: r["AUC"]) for m in MODELOS}
print("\n── Mejor configuración por modelo ──")
for m in MODELOS:
    r = mejorB[m]
    ref = REF_A.get(m, {}).get("test_auc")
    extra = f"(vs A {r['AUC']-ref:+.4f})" if ref is not None else "(sin referencia A)"
    print(f"  {m:<22}: {r['config']:<24} AUC={r['AUC']:.4f} {extra}")

### B.2 La mejor de cada modelo

In [ ]:
mejoresB = pd.DataFrame([mejorB[m] for m in MODELOS])
mejoresB = mejoresB[["modelo","config","AUC","Gini","KS","AUC_PR","Brier","F1","recall","precision","MCC"]]
display(mejoresB)
mejoresB.to_csv(OUTPUT_DIR / "B_mejores_configs.csv", sep=";", encoding="utf-8-sig", index=False)

# Gráfico: configuraciones del modelo interno contra la referencia de A
ref_int = REF_A.get("Interno", {}).get("test_auc")
cfgs = [r["config"] for r in resultadosB["Interno"]]
aucs = [r["AUC"] for r in resultadosB["Interno"]]
fig, ax = plt.subplots(figsize=(10,4.5))
colores = ["#70AD47" if (ref_int is not None and a > ref_int) else "#4472C4" for a in aucs]
b = ax.bar(cfgs, aucs, color=colores)
if ref_int is not None:
    ax.axhline(ref_int, color="red", linestyle="--", label=f"Bloque A ({ref_int})")
    ax.legend()
lo = min(aucs + ([ref_int] if ref_int else [])); hi = max(aucs + ([ref_int] if ref_int else []))
ax.set_ylim(lo-0.005, hi+0.005); ax.set_ylabel("AUC")
ax.set_title("Bloque B — modelo interno, AUC por configuración (verde = supera a A)")
for r,v in zip(b,aucs): ax.text(r.get_x()+r.get_width()/2, v+0.0005, f"{v:.4f}", ha="center", fontsize=9)
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

graficar_matrices([(POBLACIONES[m][3], predB[m][mejorB[m]["config"]], f"{m} · {mejorB[m]['config']}")
                   for m in MODELOS])

---

# Bloque C — Búsqueda amplia con validación honesta

**Objetivo.** Responder de una vez si los hiperparámetros todavía tienen algo que dar, o si A ya estaba en el techo. Para eso se abre el espacio de búsqueda hasta zonas que rozan el sobreajuste (árboles de hasta 12 niveles, regularización desde cero, hasta 120 combinaciones) y se exige mucho antes de declarar un ganador.

La regla de oro: **el juez es la validación cruzada sobre train, no el test.** Si uno elige la configuración mirando el test, termina eligiendo la que tuvo suerte con ese test en particular.

Para declarar una mejora real tienen que pasar tres cosas:

1. Que el AUC de validación cruzada supere al de A, comparando validación cruzada con validación cruzada.
2. Que esa diferencia supere el ruido, es decir, una desviación de la validación cruzada de A.
3. Que se sostenga al repetir el entrenamiento con tres semillas distintas de partición.

Al final hay una demostración de sobreajuste, marcada como tal. No es parte del método: sirve para mostrar por qué el criterio tiene que ser la validación cruzada.

In [ ]:
PARAM_DIST_C = {
    "clf__n_estimators":     randint(200, 1500),
    "clf__max_depth":        randint(3, 13),        # hasta 12 niveles
    "clf__learning_rate":    uniform(0.005, 0.295), # hasta 0.30
    "clf__subsample":        uniform(0.5, 0.5),
    "clf__colsample_bytree": uniform(0.5, 0.5),
    "clf__min_child_weight": randint(1, 20),
    "clf__gamma":            uniform(0.0, 1.0),
    "clf__reg_lambda":       uniform(0.0, 10.0),    # desde 0: sin regularización L2
    "clf__reg_alpha":        uniform(0.0, 3.0),
    "clf__max_delta_step":   randint(0, 6),
}

REF_A = globals().get("REF_A", {})
if not REF_A:
    print("Sin bloque A: las búsquedas corren igual, pero sin veredicto contra una referencia.")

resC, baseC = {}, {}

### C.1 Modelo interno

Aviso honesto: con 120 combinaciones esta búsqueda sobre la cartera completa puede tardar días. Si solo quieres probar, baja `N_ITER_C`.

In [ ]:
resC["Interno"] = correr_busqueda("Interno", POBLACIONES["Interno"], PARAM_DIST_C, N_ITER_C,
                                  "C · Interno", ref=REF_A.get("Interno"))
print("\nMétricas test:", resC["Interno"]["m"])

### C.2 Datacrédito y su línea base

In [ ]:
baseC["Datacrédito"] = correr_busqueda("Datacrédito", BASES["Datacrédito"], PARAM_DIST_C, N_ITER_C,
                                       "C · Solo internas (población DC)")
resC["Datacrédito"]  = correr_busqueda("Datacrédito", POBLACIONES["Datacrédito"], PARAM_DIST_C, N_ITER_C,
                                       "C · Datacrédito", ref=REF_A.get("Datacrédito"))
print("\nMétricas test:", resC["Datacrédito"]["m"])
print(f">>> Aporte de Datacrédito: {resC['Datacrédito']['m']['AUC'] - baseC['Datacrédito']['m']['AUC']:+.4f}")

### C.3 Vinculación digital y su línea base

In [ ]:
baseC["Vinculación Digital"] = correr_busqueda("Vinculación Digital", BASES["Vinculación Digital"],
                                               PARAM_DIST_C, N_ITER_C, "C · Solo internas (segmento VD)")
resC["Vinculación Digital"]  = correr_busqueda("Vinculación Digital", POBLACIONES["Vinculación Digital"],
                                               PARAM_DIST_C, N_ITER_C, "C · Vinculación Digital",
                                               ref=REF_A.get("Vinculación Digital"))
print("\nMétricas test:", resC["Vinculación Digital"]["m"])
print(f">>> Aporte de vinculación digital: "
      f"{resC['Vinculación Digital']['m']['AUC'] - baseC['Vinculación Digital']['m']['AUC']:+.4f}")

### C.4 ¿La mejora se sostiene con otras semillas?

Toma la mejor configuración de cada modelo, elegida por validación cruzada, y la reentrena en tres particiones distintas de train. Una mejora real aparece en todas; si aparece en unas y desaparece en otras, era ruido.

El criterio es exigente a propósito: el promedio menos una desviación tiene que quedar por encima del AUC de test de A.

In [ ]:
diagC = pd.DataFrame([resC[m]["diag"] for m in MODELOS])
cols_front = ["modelo","cv_auc","cv_std","delta_cv_vs_ref","supera_ruido","train_auc","gap_train_cv","minutos"]
otras = [c for c in diagC.columns if c not in cols_front]
diagC = diagC[[c for c in cols_front if c in diagC.columns] + otras]
display(diagC)
diagC.to_csv(OUTPUT_DIR / "C_busqueda_amplia.csv", sep=";", encoding="utf-8-sig", index=False)

val_rows = []
for m in MODELOS:
    Xtr, ytr = POBLACIONES[m][0], POBLACIONES[m][1]
    media, desv, aucs = validar_multiseed(resC[m]["estimador"], Xtr, ytr)
    ref_test = REF_A.get(m, {}).get("test_auc")
    if ref_test is None:
        veredicto = "sin ref."
    else:
        veredicto = "SI" if (media - desv) > ref_test else "no"
    print(f"\n  {m}:")
    print(f"    AUC en 3 semillas: {[round(a,4) for a in aucs]}")
    print(f"    Promedio {media:.4f} ± {desv:.4f}  |  A test = {ref_test}")
    print(f"    {'Mejora sostenida' if veredicto == 'SI' else 'No se sostiene o no hay referencia'}")
    val_rows.append({"modelo": m, "auc_medio_multiseed": round(media,4), "std_multiseed": round(desv,4),
                     "auc_A_test": ref_test, "mejora_sostenida": veredicto})

validacionC = pd.DataFrame(val_rows)
display(validacionC)
validacionC.to_csv(OUTPUT_DIR / "C_validacion_multiseed.csv", sep=";", encoding="utf-8-sig", index=False)

### C.5 Veredicto

Junta los dos filtros. Las métricas de test aparecen solo como referencia: no son el criterio.

In [ ]:
resumen_testC = pd.DataFrame([resC[m]["m"] for m in MODELOS])[["modelo","AUC","Gini","KS","AUC_PR","Brier"]]
display(resumen_testC)

n_ruido = int((diagC["supera_ruido"] == "SI").sum()) if "supera_ruido" in diagC else 0
n_sost  = int((validacionC["mejora_sostenida"] == "SI").sum())
print(f"Modelos que superan el ruido en validación cruzada: {n_ruido} de {len(MODELOS)}")
print(f"Modelos con mejora sostenida en varias semillas:  {n_sost} de {len(MODELOS)}")

if not REF_A:
    print("\nSin bloque A no hay contra qué emitir veredicto.")
elif n_sost == 0:
    print("\nLa búsqueda amplia no encontró una mejora real sobre el bloque A.")
    print("Si el AUC de test sube un poco, no se sostiene entre semillas: es ruido.")
else:
    print(f"\n{n_sost} modelo(s) muestran mejora sostenida. Antes de adoptarla,")
    print("conviene confirmarla con más semillas.")

veredictoC = validacionC.merge(
    diagC[[c for c in ["modelo","cv_auc","delta_cv_vs_ref","supera_ruido","gap_train_cv"] if c in diagC.columns]],
    on="modelo", how="left")
veredictoC.to_csv(OUTPUT_DIR / "C_veredicto.csv", sep=";", encoding="utf-8-sig", index=False)

### C.6 Demostración de sobreajuste (ilustrativa)

Esta sección no es metodológica. Entrena el modelo interno sin ningún freno (sin regularización, `min_child_weight=1`) con profundidades crecientes, y compara el AUC de train, de validación cruzada y de test. La idea es ver con los propios datos cómo se ve el sobreajuste: el train sube y se aleja, mientras validación cruzada y test se quedan quietos o bajan. Esa brecha es la razón por la que en C el juez fue la validación cruzada.

In [ ]:
prof_grid = [3, 5, 7, 9, 11, 13, 15]
cv_demo = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
spw_d = (ytr_i==0).sum()/(ytr_i==1).sum()

demo = []
for prof in prof_grid:
    prep = construir_preprocesador(num_i, cat_i)
    clf = XGBClassifier(n_estimators=400, max_depth=prof, learning_rate=0.1,
                        min_child_weight=1, reg_lambda=0.0, gamma=0.0,   # sin frenos
                        scale_pos_weight=spw_d, eval_metric="auc", tree_method="hist",
                        random_state=RANDOM_STATE, n_jobs=-1)
    pipe = Pipeline([("prep", prep), ("clf", clf)])
    pipe.fit(Xtr_i, ytr_i)
    auc_train = roc_auc_score(ytr_i, pipe.predict_proba(Xtr_i)[:,1])
    auc_test  = roc_auc_score(yte_i, pipe.predict_proba(Xte_i)[:,1])
    auc_cv    = cross_val_score(pipe, Xtr_i, ytr_i, cv=cv_demo, scoring="roc_auc", n_jobs=1).mean()
    demo.append({"max_depth": prof, "auc_train": round(auc_train,4), "auc_cv": round(auc_cv,4),
                 "auc_test": round(auc_test,4), "gap_train_cv": round(auc_train-auc_cv,4)})
    print(f"  depth={prof:>2}: train={auc_train:.4f}  CV={auc_cv:.4f}  test={auc_test:.4f}  "
          f"gap={auc_train-auc_cv:.4f}")

demo_df = pd.DataFrame(demo)
demo_df.to_csv(OUTPUT_DIR / "C_demo_sobreajuste.csv", sep=";", encoding="utf-8-sig", index=False)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(demo_df["max_depth"], demo_df["auc_train"], "o-", label="AUC train", color="#C00000", linewidth=2)
ax.plot(demo_df["max_depth"], demo_df["auc_cv"],    "s-", label="AUC validación cruzada", color="#2E75B6", linewidth=2)
ax.plot(demo_df["max_depth"], demo_df["auc_test"],  "^-", label="AUC test", color="#70AD47", linewidth=2)
ax.set_xlabel("max_depth (profundidad del árbol)"); ax.set_ylabel("AUC")
ax.set_title("Firma del sobreajuste: train se aleja, validación y test no lo siguen")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()